# 🕸️ Session 2 — The Cross-Company Risk Exposure Graph

In Session 1, our RAG system stumbled on *"which companies share exposure to export control
risk, and how are they connected?"* — a question about **relationships**, not passages.

Now we build the structure that answers it: a **knowledge graph** of 8 semiconductor
companies extracted from their SEC filings.

```
                      DISCLOSES
   (Company) ───────────────────────► (RiskFactor)
       │                                  e.g. "US export controls on advanced chips"
       │  SUBJECT_TO
       ├────────────────► (Regulation)
       │                     e.g. "U.S. Export Administration Regulations"
       │  DEPENDS_ON
       └────────────────► (Company)
                             e.g. NVIDIA ──DEPENDS_ON──► TSMC
```

**The cast:** NVIDIA, AMD, Qualcomm, Micron, Intel, Broadcom (10-K filers) — plus TSMC and
ASML, who file the foreign equivalent (20-F) and sit at the heart of everyone's supply chain.


In [ ]:
%pip install -q neo4j pandas pyvis

## Step 1 — The pre-extracted graph data

Entity/relationship extraction from the filings was done **before the session** (LLM-assisted
triple extraction over each filing's Item 1A "Risk Factors" section — see
[`scripts/extract_graph_data.py`](https://github.com/YOUR_GITHUB_USERNAME/finance-ai-rag-kg/blob/main/scripts/extract_graph_data.py)
— then hand-checked against the filings). Live LLM extraction is too slow and too flaky for a
30-minute slot; in production you'd run it as a batch pipeline the same way.

The result is four small CSVs — **nodes** (companies, risk factors, regulations) and **edges**
(relationships, each carrying an `evidence` note pointing back at the filing that supports it).


In [ ]:
from pathlib import Path

import pandas as pd

GITHUB_REPO = "YOUR_GITHUB_USERNAME/finance-ai-rag-kg"  # ← update after pushing your fork


def load_csv(name):
    for base in (Path("../data/graph"), Path("data/graph")):
        if (base / name).exists():
            return pd.read_csv(base / name)
    return pd.read_csv(f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/data/graph/{name}")


companies = load_csv("companies.csv")
risk_factors = load_csv("risk_factors.csv")
regulations = load_csv("regulations.csv")
relationships = load_csv("relationships.csv")

print(f"{len(companies)} companies · {len(risk_factors)} risk factors · "
      f"{len(regulations)} regulations · {len(relationships)} relationships")
display(companies)
display(relationships.sample(8, random_state=7))

## Step 2 — Connect to Neo4j

We use **Neo4j Aura** (free tier) — a real graph database in the cloud, with the Cypher query
language and a browser-based graph visualizer.

> 🔑 **Before the session** you created a free instance at
> [console.neo4j.io](https://console.neo4j.io) and saved the connection URI + password
> (Aura shows them exactly once, as a downloadable `.txt`). Paste them below —
> or in Colab, store `NEO4J_URI` / `NEO4J_PASSWORD` as secrets (🔑 sidebar icon).


In [ ]:
import os

from neo4j import GraphDatabase


def get_credential(name, prompt_text, secret=False):
    if os.environ.get(name):
        return os.environ[name]
    try:  # Colab secret
        from google.colab import userdata

        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    try:
        import getpass

        return (getpass.getpass(prompt_text) if secret else input(prompt_text)).strip()
    except Exception:
        return ""


NEO4J_URI = get_credential("NEO4J_URI", "Neo4j URI (neo4j+s://xxxx.databases.neo4j.io): ")
NEO4J_USER = get_credential("NEO4J_USERNAME", "Username [neo4j]: ") or "neo4j"
NEO4J_PASSWORD = get_credential("NEO4J_PASSWORD", "Password: ", secret=True)

driver = None
if NEO4J_URI and NEO4J_PASSWORD:
    try:
        driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
        driver.verify_connectivity()
        print("Connected to Neo4j ✅")
    except Exception as err:
        driver = None
        print(f"⚠️ Connection failed: {err}")
else:
    print("⚠️ No credentials — Cypher cells will be skipped (the PyVis visualization further down still works).")


def run_query(cypher, **params):
    """Run Cypher and return the results as a DataFrame."""
    if driver is None:
        print("⚠️ Not connected to Neo4j — skipping.")
        return None
    records, _, _ = driver.execute_query(cypher, **params)
    return pd.DataFrame([record.data() for record in records])

## Step 3 — Load the graph

Three ideas to notice in the Cypher below:

- **Constraints** make node keys unique (and fast to look up) — like a primary key.
- **`MERGE`** means *create unless it already exists* — so re-running this cell is harmless.
- **`UNWIND $rows`** turns our DataFrame rows into one batched, parameterized statement per
  node/edge type — no string-gluing data into queries.


In [ ]:
if driver:
    for constraint in [
        "CREATE CONSTRAINT company_ticker IF NOT EXISTS FOR (c:Company) REQUIRE c.ticker IS UNIQUE",
        "CREATE CONSTRAINT risk_factor_id IF NOT EXISTS FOR (r:RiskFactor) REQUIRE r.risk_id IS UNIQUE",
        "CREATE CONSTRAINT regulation_id IF NOT EXISTS FOR (g:Regulation) REQUIRE g.reg_id IS UNIQUE",
    ]:
        driver.execute_query(constraint)

    run_query("""
        UNWIND $rows AS row
        MERGE (c:Company {ticker: row.ticker})
        SET c.name = row.name, c.country = row.country, c.filing_form = row.filing_form
        """, rows=companies.to_dict("records"))

    run_query("""
        UNWIND $rows AS row
        MERGE (r:RiskFactor {risk_id: row.risk_id})
        SET r.name = row.name, r.description = row.description
        """, rows=risk_factors.to_dict("records"))

    run_query("""
        UNWIND $rows AS row
        MERGE (g:Regulation {reg_id: row.reg_id})
        SET g.name = row.name, g.description = row.description
        """, rows=regulations.to_dict("records"))

    # one batched MERGE per relationship type (labels/types can't be parameters in Cypher)
    for rel_type, target_label, target_key in [
        ("DISCLOSES", "RiskFactor", "risk_id"),
        ("SUBJECT_TO", "Regulation", "reg_id"),
        ("DEPENDS_ON", "Company", "ticker"),
    ]:
        rows = relationships[relationships.rel_type == rel_type].to_dict("records")
        run_query(f"""
            UNWIND $rows AS row
            MATCH (source:Company {{ticker: row.source}})
            MATCH (target:{target_label} {{{target_key}: row.target}})
            MERGE (source)-[edge:{rel_type}]->(target)
            SET edge.evidence = row.evidence
            """, rows=rows)

    display(run_query("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS nodes ORDER BY nodes DESC"))
    display(run_query("MATCH ()-[e]->() RETURN type(e) AS relationship, count(*) AS edges ORDER BY edges DESC"))

## Step 4 — Warm-up: what RAG could already do

One company's risks — a per-document question. Note the `evidence` property carried on
every edge: the graph stays **auditable**, each edge points back at filing text.


In [ ]:
display(run_query("""
    MATCH (c:Company {ticker: 'NVDA'})-[d:DISCLOSES]->(r:RiskFactor)
    RETURN r.name AS risk, d.evidence AS evidence
"""))

## Step 5 — 💥 Payoff #1: the question RAG couldn't answer

*"Which companies share exposure to the same risk?"* — one `MATCH`, one `collect`:


In [ ]:
display(run_query("""
    MATCH (c:Company)-[:DISCLOSES]->(r:RiskFactor)
    WITH r, collect(c.ticker) AS exposed
    WHERE size(exposed) >= 4
    RETURN r.name AS shared_risk, size(exposed) AS n_companies, exposed
    ORDER BY n_companies DESC
"""))

All eight companies — American, Taiwanese, Dutch — connect through the *same* export-control
risk node. In Session 1 that fact was smeared across three separate filings in three different
phrasings. Here it's **one node with eight edges**.

## Step 6 — 💥 Payoff #2: hidden exposure through the supply chain

Who is exposed to *Taiwan concentration risk*? Some companies disclose it themselves. Others
never mention it prominently — but **depend on a company that does**. That second group is
invisible to per-document analysis:


In [ ]:
display(run_query("""
    MATCH (taiwan:RiskFactor {risk_id: 'RF_TAIWAN_CONCENTRATION'})
    MATCH (c:Company)
    WHERE (c)-[:DISCLOSES]->(taiwan)
       OR (c)-[:DEPENDS_ON]->(:Company)-[:DISCLOSES]->(taiwan)
    RETURN c.ticker AS company,
           EXISTS { (c)-[:DISCLOSES]->(taiwan) } AS discloses_it_directly,
           [ (c)-[:DEPENDS_ON]->(s:Company)-[:DISCLOSES]->(taiwan) | s.ticker ] AS via_suppliers
    ORDER BY discloses_it_directly DESC, company
"""))

👀 Look at the rows where `discloses_it_directly` is `false`: exposure that appears **nowhere
in that company's own filing** as a Taiwan risk — it emerges only when you connect documents.
This is the AWS-style result: *risk concentration that's invisible in per-document data.*

## Step 7 — 💥 Payoff #3: single points of failure

Flip the direction: which company, if stressed, touches the most others?


In [ ]:
display(run_query("""
    MATCH (supplier:Company)<-[:DEPENDS_ON]-(dependent:Company)
    WITH supplier, collect(dependent.ticker) AS dependents
    OPTIONAL MATCH (supplier)-[:DISCLOSES]->(r:RiskFactor)
    RETURN supplier.ticker AS chokepoint,
           size(dependents) AS n_dependents,
           dependents,
           collect(r.name) AS the_chokepoint_own_risks
    ORDER BY n_dependents DESC
"""))

One company sits under five of the others — and its *own* top risks are Taiwan concentration
and export controls. The concentration the market worries about isn't in any single filing;
it's in the **shape of the graph**.

## Step 8 — 👁️ See it

Numbers in tables still under-sell this. Let's *look* at the graph (built straight from the
CSVs, so this works even without Neo4j):


In [ ]:
from IPython.display import HTML
from pyvis.network import Network

net = Network(height="650px", width="100%", directed=True, cdn_resources="in_line", notebook=False)
net.barnes_hut(gravity=-3000, spring_length=140)

STYLE = {"Company": ("#4C8BF5", 28), "RiskFactor": ("#F5A34C", 22), "Regulation": ("#9B59B6", 20)}

for _, row in companies.iterrows():
    color, size = STYLE["Company"]
    net.add_node(row.ticker, label=row.ticker, title=row["name"], color=color, size=size, shape="dot")
for _, row in risk_factors.iterrows():
    color, size = STYLE["RiskFactor"]
    net.add_node(row.risk_id, label=row["name"], title=row.description, color=color, size=size, shape="diamond")
for _, row in regulations.iterrows():
    color, size = STYLE["Regulation"]
    net.add_node(row.reg_id, label=row["name"], title=row.description, color=color, size=size, shape="triangle")

EDGE_COLOR = {"DISCLOSES": "#F5A34C", "SUBJECT_TO": "#9B59B6", "DEPENDS_ON": "#4C8BF5"}
for _, row in relationships.iterrows():
    net.add_edge(row.source, row.target, title=f"{row.rel_type}: {row.evidence}",
                 color=EDGE_COLOR[row.rel_type], width=2.5 if row.rel_type == "DEPENDS_ON" else 1)

HTML(net.generate_html())

🔵 companies · 🔶 risk factors · 🟣 regulations — hover any edge to see the filing evidence
behind it. Drag nodes around: notice how TSMC and the export-controls risk sit at the
gravitational center of the sector.

### The full experience: Neo4j Browser

On your Aura console, hit **Open** (Query tab) and paste:

```cypher
MATCH path = (:Company)-[]->() RETURN path
```

…and you get the same graph, live from the database, interactively explorable. For just the
shared-risk picture:

```cypher
MATCH path = (c:Company)-[:DISCLOSES]->(r:RiskFactor {risk_id: 'RF_EXPORT_CONTROLS'})
RETURN path
```


## Stretch — close the loop: English → Cypher → graph 🤖

Session 1's LLM answered from retrieved *text*. It can also answer from the *graph* — by
writing Cypher for us. (Uses the same Gemini key as Session 1.)


In [ ]:
GRAPH_SCHEMA = """Nodes:
  (:Company {ticker, name, country, filing_form})
  (:RiskFactor {risk_id, name, description})
  (:Regulation {reg_id, name, description})
Relationships (all edges have an `evidence` string property):
  (:Company)-[:DISCLOSES]->(:RiskFactor)
  (:Company)-[:SUBJECT_TO]->(:Regulation)
  (:Company)-[:DEPENDS_ON]->(:Company)"""


def get_gemini_key():
    if os.environ.get("GEMINI_API_KEY"):
        return os.environ["GEMINI_API_KEY"]
    try:
        from google.colab import userdata

        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return ""


def ask_the_graph(question):
    key = get_gemini_key()
    if not key:
        print("⚠️ No GEMINI_API_KEY available — skipping text-to-Cypher.")
        return
    from google import genai

    client = genai.Client(api_key=key)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=(
            "Translate the question into a single read-only Cypher query for this Neo4j schema.\n"
            f"{GRAPH_SCHEMA}\n\nReturn ONLY the Cypher, no explanation, no code fences.\n"
            f"Question: {question}"
        ),
    )
    cypher = response.text.strip().removeprefix("```cypher").removeprefix("```").removesuffix("```").strip()
    print(f"❓ {question}\n\n🤖 generated Cypher:\n{cypher}\n")
    result = run_query(cypher)
    if result is not None:
        display(result)


ask_the_graph("Which regulations affect the most companies?")

In [ ]:
ask_the_graph("Which companies are exposed to export control risk without being subject to the US Export Administration Regulations... wait, are there any?")  # ← change me!

## Where you'd take this next 🚀

- **Scale the extraction**: `scripts/extract_graph_data.py` shows the LLM-assisted triple
  extraction — run it over all S&P 500 filings, every quarter.
- **Add node types**: officers, auditors, customers, geographies → circular-ownership and
  governance patterns (the AWS write-up's original result).
- **Combine with Session 1**: GraphRAG — retrieve *subgraphs* as LLM context instead of
  (or alongside) text chunks, so the model sees relationships and prose.
- **Point it at time**: load each year's filings as snapshots and watch risk clusters *form*
  (could you have seen the 2023 regional-bank CRE cluster early? that's a graph query).

*Both notebooks + all data: `github.com/YOUR_GITHUB_USERNAME/finance-ai-rag-kg`*
